In [ ]:
# Extract capture dates/times from all PCAP/PCAPNG files in Google Drive
# Input:  /content/drive/MyDrive/mobile
# Output: /content/drive/MyDrive/mobile_capture_timeline

from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update
!apt-get -qq install -y tshark

from pathlib import Path
import subprocess
import re
import pandas as pd
from datetime import datetime

INPUT_DIR = Path("/content/drive/MyDrive/mobile")
OUTPUT_DIR = Path("/content/drive/MyDrive/mobile_capture_timeline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def parse_name(filename):
    stem = Path(filename).stem.lower()
    m = re.search(r"(.+?)[_-]day[_-]?(\d+)", stem)
    if m:
        return m.group(1).replace("_", " ").title(), int(m.group(2))
    return stem.title(), None

def get_capture_times(pcap):
    # Read only the first packet timestamp.
    first_cmd = [
        "tshark", "-r", str(pcap),
        "-T", "fields", "-e", "frame.time_epoch",
        "-c", "1"
    ]
    first = subprocess.run(first_cmd, capture_output=True, text=True, check=True)
    first_lines = [x.strip() for x in first.stdout.splitlines() if x.strip()]
    if not first_lines:
        return None

    # capinfos provides first/last packet time efficiently without dumping all packets.
    info_cmd = ["capinfos", "-a", "-e", "-u", "-c", str(pcap)]
    info = subprocess.run(info_cmd, capture_output=True, text=True, check=True)

    first_epoch = float(first_lines[0])

    # Obtain last packet epoch using tshark; tail is handled by Python.
    last_cmd = [
        "tshark", "-r", str(pcap),
        "-T", "fields", "-e", "frame.time_epoch"
    ]
    last = subprocess.run(last_cmd, capture_output=True, text=True, check=True)
    last_lines = [x.strip() for x in last.stdout.splitlines() if x.strip()]
    last_epoch = float(last_lines[-1])

    return first_epoch, last_epoch, len(last_lines)

pcaps = sorted(list(INPUT_DIR.rglob("*.pcap")) + list(INPUT_DIR.rglob("*.pcapng")))
print(f"Found {len(pcaps)} captures")

rows = []
errors = []

for i, pcap in enumerate(pcaps, 1):
    print(f"[{i}/{len(pcaps)}] {pcap.name}")
    try:
        result = get_capture_times(pcap)
        if result is None:
            errors.append({"capture_file": pcap.name, "error": "No packets"})
            continue

        first_epoch, last_epoch, packet_count = result
        app, day = parse_name(pcap.name)

        start = datetime.fromtimestamp(first_epoch)
        end = datetime.fromtimestamp(last_epoch)

        rows.append({
            "capture_file": pcap.name,
            "application": app,
            "day": day,
            "capture_date": start.date().isoformat(),
            "start_datetime": start.isoformat(sep=" "),
            "end_datetime": end.isoformat(sep=" "),
            "duration_seconds": last_epoch - first_epoch,
            "duration_minutes": (last_epoch - first_epoch) / 60.0,
            "packet_count": packet_count,
            "first_packet_epoch": first_epoch,
            "last_packet_epoch": last_epoch,
        })

    except Exception as e:
        errors.append({"capture_file": pcap.name, "error": str(e)})

df = pd.DataFrame(rows)

if not df.empty:
    df = df.sort_values(["application", "day", "start_datetime"])
    df.to_csv(OUTPUT_DIR / "capture_timeline.csv", index=False)

    # Per-application Day 1 -> Day 5 temporal span.
    spans = []
    for app, g in df.groupby("application"):
        g = g.dropna(subset=["day"]).sort_values("day")
        d1 = g[g["day"] == 1]
        d5 = g[g["day"] == 5]

        if not d1.empty and not d5.empty:
            start1 = pd.to_datetime(d1.iloc[0]["start_datetime"])
            start5 = pd.to_datetime(d5.iloc[0]["start_datetime"])
            elapsed = start5 - start1

            dates = g["capture_date"].tolist()
            parsed_dates = pd.to_datetime(g["capture_date"]).sort_values()
            gaps = parsed_dates.diff().dropna().dt.days.tolist()

            spans.append({
                "application": app,
                "day1_date": d1.iloc[0]["capture_date"],
                "day5_date": d5.iloc[0]["capture_date"],
                "day1_to_day5_elapsed_hours": elapsed.total_seconds() / 3600,
                "day1_to_day5_elapsed_days": elapsed.total_seconds() / 86400,
                "capture_dates": "; ".join(dates),
                "calendar_day_gaps": "; ".join(map(str, gaps)),
                "consecutive_calendar_dates": all(x == 1 for x in gaps) if gaps else False,
            })

    span_df = pd.DataFrame(spans)
    span_df.to_csv(OUTPUT_DIR / "application_temporal_span.csv", index=False)

    # Dataset-wide collection period.
    dataset_start = pd.to_datetime(df["start_datetime"]).min()
    dataset_end = pd.to_datetime(df["end_datetime"]).max()

    overall = pd.DataFrame([{
        "number_of_captures": len(df),
        "dataset_first_capture": dataset_start,
        "dataset_last_capture": dataset_end,
        "overall_elapsed_hours": (dataset_end - dataset_start).total_seconds() / 3600,
        "overall_elapsed_days": (dataset_end - dataset_start).total_seconds() / 86400,
        "unique_capture_dates": df["capture_date"].nunique(),
        "capture_dates": "; ".join(sorted(df["capture_date"].unique()))
    }])
    overall.to_csv(OUTPUT_DIR / "dataset_temporal_summary.csv", index=False)

if errors:
    pd.DataFrame(errors).to_csv(OUTPUT_DIR / "timeline_errors.csv", index=False)

print("\nDone.")
print("Results:", OUTPUT_DIR)
print("  capture_timeline.csv")
print("  application_temporal_span.csv")
print("  dataset_temporal_summary.csv")
if errors:
    print("  timeline_errors.csv")
